# ELB / Shadow-Rate Basics\n\nThis notebook demonstrates effective lower bound (ELB) handling via latent shadow-rate\ndata augmentation.\n\nKey idea: the observed policy rate is censored at the ELB, but the latent shadow rate\ncontinues to move below the bound.\n

In [ ]:
import numpy as np\n\nfrom srvar import Dataset, ElbSpec\nfrom srvar.api import fit, forecast\nfrom srvar.spec import ModelSpec, PriorSpec, SamplerConfig\n\nrng = np.random.default_rng(0)\nt = 80\nbound = 0.0\n\n# Latent policy rate can go below the ELB, but observed rate is floored.\nr_latent = rng.standard_normal(t) * 0.05\nr_obs = np.maximum(r_latent, bound)\ny2 = rng.standard_normal(t)\n\nds = Dataset.from_arrays(values=np.column_stack([r_obs, y2]), variables=["r", "y2"])\n\nmodel = ModelSpec(\n    p=2,\n    include_intercept=True,\n    elb=ElbSpec(bound=bound, applies_to=["r"]),\n)\nprior = PriorSpec.niw_default(k=1 + ds.N * model.p, n=ds.N)\nsampler = SamplerConfig(draws=120, burn_in=20, thin=2)\n\nfit_res = fit(ds, model, prior, sampler, rng=np.random.default_rng(1))\nfit_res.latent_dataset.values[:5]\n

In [ ]:
fc = forecast(fit_res, horizons=[1, 4, 8], draws=200, rng=np.random.default_rng(2))\nprint("observed draws shape:", fc.draws.shape)\nprint("latent draws shape:", None if fc.latent_draws is None else fc.latent_draws.shape)\n\nr_idx = ds.variables.index("r")\nprint("min observed forecast r:", fc.draws[..., r_idx].min())\nif fc.latent_draws is not None:\n    print("min latent forecast r:", fc.latent_draws[..., r_idx].min())\n

## Labeled outputs (optional)\n\nIf you install `.[xarray]`, you can convert results to labeled `xarray.Dataset` objects.\n

In [ ]:
try:\n    from srvar.xarray import fit_to_xarray, forecast_to_xarray\n\n    display(fit_to_xarray(fit_res))\n    display(forecast_to_xarray(fc))\nexcept ImportError as e:\n    print(e)\n